In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

BASE = "/content/drive/MyDrive/rgb_dataset"

for root, dirs, files in os.walk(BASE):
    print(root, len(files))

/content/drive/MyDrive/rgb_dataset 1
/content/drive/MyDrive/rgb_dataset/test 0
/content/drive/MyDrive/rgb_dataset/test/images 254
/content/drive/MyDrive/rgb_dataset/test/masks 254
/content/drive/MyDrive/rgb_dataset/train 0
/content/drive/MyDrive/rgb_dataset/train/images 811
/content/drive/MyDrive/rgb_dataset/train/masks 811
/content/drive/MyDrive/rgb_dataset/val 0
/content/drive/MyDrive/rgb_dataset/val/images 203
/content/drive/MyDrive/rgb_dataset/val/masks 203


In [3]:
BASE = "/content/drive/MyDrive/rgb_dataset"
IMG_SIZE = (256, 256)

In [4]:
import cv2
import numpy as np
import os
from tqdm import tqdm

def load_data(split):

    img_dir = os.path.join(BASE, split, "images")
    mask_dir = os.path.join(BASE, split, "masks")

    images = []
    masks = []

    files = sorted(os.listdir(img_dir))

    for f in tqdm(files):

        img_path = os.path.join(img_dir, f)
        mask_path = os.path.join(mask_dir, f)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)

        mask = cv2.imread(mask_path, 0)
        mask = cv2.resize(mask, IMG_SIZE)

        img = img / 255.0
        mask = (mask > 127).astype(np.float32)

        images.append(img)
        masks.append(mask[..., None])

    return np.array(images), np.array(masks)

In [5]:
X_train, Y_train = load_data("train")
X_val, Y_val     = load_data("val")
X_test, Y_test   = load_data("test")

print(X_train.shape, Y_train.shape)

  0%|          | 0/811 [00:03<?, ?it/s]


error: OpenCV(4.14.0) /io/opencv/modules/imgproc/src/resize.cpp:4014: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [ ]:
!pip install segmentation-models-pytorch albumentations -q
print("Done!")


## 📚 Imports

In [ ]:
import os, cv2, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## ⚙️ Config & Paths

In [ ]:
# ── Update this to where you extracted the Zenodo zip ──
BASE = Path("/content/drive/MyDrive/oil_spill_drone")

TRAIN_IMG  = BASE / "train"
TRAIN_MASK = BASE / "train_annot"
VAL_IMG    = BASE / "val"
VAL_MASK   = BASE / "val_annot"
TEST_IMG   = BASE / "test"
TEST_MASK  = BASE / "test_annot"

IMG_SIZE   = 256          # resize all images to 256×256
BATCH_SIZE = 16
EPOCHS     = 50
LR         = 1e-4
NUM_CLASSES = 3           # Oil=0, Water=1, Other=2
WEIGHTS_PATH = "/content/drive/MyDrive/best_oilspill_unet.pth"

# Class definitions
CLASS_NAMES  = ["Oil", "Water", "Other"]
CLASS_COLORS = [(255, 0, 0), (0, 0, 255), (128, 128, 128)]  # RGB for viz

print("Paths configured.")
for p in [TRAIN_IMG, TRAIN_MASK, VAL_IMG, VAL_MASK, TEST_IMG, TEST_MASK]:
    count = len(list(p.glob("*.png"))) if p.exists() else 0
    print(f"  {p.name}: {count} files")


## 🔍 Module 2: EDA — Understand Mask Encoding

In [ ]:
# CamVid format: masks are grayscale PNGs where pixel value = class index
# Let's verify by inspecting a sample mask
sample_imgs  = sorted(TRAIN_IMG.glob("*.png"))[:6]
sample_masks = [TRAIN_MASK / img.name for img in sample_imgs]

fig, axes = plt.subplots(3, 6, figsize=(20, 10))
for col, (img_path, msk_path) in enumerate(zip(sample_imgs, sample_masks)):
    img  = cv2.cvtColor(cv2.imread(str(img_path)),  cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE)

    # Color mask for visualization
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_idx, color in enumerate(CLASS_COLORS):
        color_mask[mask == cls_idx] = color

    axes[0, col].imshow(img);         axes[0, col].set_title(f"Image {col+1}"); axes[0, col].axis("off")
    axes[1, col].imshow(mask, cmap="tab10", vmin=0, vmax=2);
                                       axes[1, col].set_title("Raw mask"); axes[1, col].axis("off")
    axes[2, col].imshow(color_mask);   axes[2, col].set_title("Color mask"); axes[2, col].axis("off")

patches = [mpatches.Patch(color=np.array(c)/255, label=n)
           for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=12)
plt.suptitle("Sample Images & Segmentation Masks", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# Compute class pixel distribution across training set
print("Computing class distribution (this may take ~1 min)...")
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)

for msk_path in tqdm(list(TRAIN_MASK.glob("*.png"))):
    mask = cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE)
    for c in range(NUM_CLASSES):
        class_counts[c] += np.sum(mask == c)

total = class_counts.sum()
print("\n=== Class Distribution (Training Set) ===")
for i, (name, count) in enumerate(zip(CLASS_NAMES, class_counts)):
    print(f"  {name:>6}: {count:>12,} px  ({count/total*100:.2f}%)")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(CLASS_NAMES, class_counts / total * 100,
              color=[np.array(c)/255 for c in CLASS_COLORS], edgecolor="black", linewidth=1.2)
for bar, pct in zip(bars, class_counts/total*100):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{pct:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Pixel %"); ax.set_title("Training Class Distribution")
plt.tight_layout(); plt.show()

# Compute class weights for loss (inverse frequency)
class_weights = total / (NUM_CLASSES * class_counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
print("\nClass weights for loss:", np.round(class_weights, 3))
CLASS_WEIGHTS = torch.FloatTensor(class_weights).to(DEVICE)


## 🗃️ Dataset & Augmentation Pipeline

In [ ]:
class OilSpillDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_paths  = sorted(Path(img_dir).glob("*.png"))
        self.mask_dir   = Path(mask_dir)
        self.transform  = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path  = self.img_paths[idx]
        mask_path = self.mask_dir / img_path.name

        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        # Clip mask to valid range (0,1,2) — safety check
        mask = np.clip(mask, 0, NUM_CLASSES - 1)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]

        return image, mask.long()


# ── Augmentations ──
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=3, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # ImageNet stats
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# ── DataLoaders ──
train_ds = OilSpillDataset(TRAIN_IMG, TRAIN_MASK, transform=train_transform)
val_ds   = OilSpillDataset(VAL_IMG,   VAL_MASK,   transform=val_transform)
test_ds  = OilSpillDataset(TEST_IMG,  TEST_MASK,  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


In [ ]:
# Visualize augmented samples
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for col in range(6):
    img_t, mask_t = train_ds[col]
    img_np = (img_t.permute(1,2,0).numpy() * std + mean).clip(0,1)
    mask_np = mask_t.numpy()

    color_mask = np.zeros((*mask_np.shape, 3))
    for c, color in enumerate(CLASS_COLORS):
        color_mask[mask_np == c] = np.array(color)/255

    axes[0, col].imshow(img_np);     axes[0, col].set_title(f"Aug img {col+1}"); axes[0, col].axis("off")
    axes[1, col].imshow(color_mask); axes[1, col].set_title("Aug mask");          axes[1, col].axis("off")

patches = [mpatches.Patch(color=np.array(c)/255, label=n)
           for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=11)
plt.suptitle("Augmented Training Samples", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


## 🧠 Module 3: Model — U-Net + EfficientNet-B3

In [ ]:
model = smp.Unet(
    encoder_name    = "efficientnet-b3",   # pretrained ImageNet backbone
    encoder_weights = "imagenet",
    in_channels     = 3,                   # RGB
    classes         = NUM_CLASSES,
    activation      = None                 # raw logits — we apply softmax in loss
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


## 📐 Loss Functions & Metrics

In [ ]:
class CombinedLoss(nn.Module):
    """Weighted CrossEntropy + Dice Loss — handles class imbalance well."""
    def __init__(self, class_weights=None, dice_weight=0.5):
        super().__init__()
        self.ce        = nn.CrossEntropyLoss(weight=class_weights)
        self.dice_w    = dice_weight

    def dice_loss(self, logits, targets, smooth=1.0):
        probs  = F.softmax(logits, dim=1)              # (B, C, H, W)
        one_hot = F.one_hot(targets, NUM_CLASSES).permute(0, 3, 1, 2).float()
        inter  = (probs * one_hot).sum(dim=(2, 3))
        denom  = probs.sum(dim=(2, 3)) + one_hot.sum(dim=(2, 3))
        dice   = (2 * inter + smooth) / (denom + smooth)
        return 1 - dice.mean()

    def forward(self, logits, targets):
        return (1 - self.dice_w) * self.ce(logits, targets) + self.dice_w * self.dice_loss(logits, targets)


def compute_metrics(preds, targets, num_classes=NUM_CLASSES):
    """Per-class IoU, Dice, Precision, Recall + overall Pixel Accuracy."""
    preds   = preds.flatten()
    targets = targets.flatten()
    acc     = (preds == targets).float().mean().item()

    iou_list, dice_list, prec_list, rec_list = [], [], [], []
    for c in range(num_classes):
        tp = ((preds == c) & (targets == c)).sum().float()
        fp = ((preds == c) & (targets != c)).sum().float()
        fn = ((preds != c) & (targets == c)).sum().float()
        iou  = tp / (tp + fp + fn + 1e-6)
        dice = 2*tp / (2*tp + fp + fn + 1e-6)
        prec = tp / (tp + fp + 1e-6)
        rec  = tp / (tp + fn + 1e-6)
        iou_list.append(iou.item()); dice_list.append(dice.item())
        prec_list.append(prec.item()); rec_list.append(rec.item())

    return {
        "pixel_acc": acc,
        "mean_iou":  np.mean(iou_list),
        "mean_dice": np.mean(dice_list),
        "iou":       iou_list,
        "dice":      dice_list,
        "precision": prec_list,
        "recall":    rec_list
    }

criterion  = CombinedLoss(class_weights=CLASS_WEIGHTS, dice_weight=0.5)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print("Loss, optimizer, scheduler ready.")


## 🏋️ Module 4: Training

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    all_preds, all_targets = [], []

    for imgs, masks in tqdm(loader, desc="Train", leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu()); all_targets.append(masks.cpu())

    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    metrics = compute_metrics(all_preds, all_targets)
    metrics["loss"] = total_loss / len(loader)
    return metrics


def val_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Val  ", leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            total_loss += criterion(logits, masks).item()
            preds = logits.argmax(dim=1)
            all_preds.append(preds.cpu()); all_targets.append(masks.cpu())

    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    metrics = compute_metrics(all_preds, all_targets)
    metrics["loss"] = total_loss / len(loader)
    return metrics


In [ ]:
history = {"train": [], "val": []}
best_iou   = 0.0
patience   = 12
no_improve = 0

print(f"Training for up to {EPOCHS} epochs (early stop patience={patience})\n")
print(f"{'Epoch':>5} | {'T-Loss':>7} | {'T-IoU':>7} | {'T-Dice':>7} | {'V-Loss':>7} | {'V-IoU':>7} | {'V-Dice':>7} | {'LR':>9}")
print("-" * 80)

for epoch in range(1, EPOCHS + 1):
    t = train_epoch(model, train_loader, criterion, optimizer)
    v = val_epoch(model,   val_loader,   criterion)
    scheduler.step()

    history["train"].append(t)
    history["val"].append(v)

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"{epoch:>5} | {t['loss']:>7.4f} | {t['mean_iou']:>7.4f} | {t['mean_dice']:>7.4f} | "
          f"{v['loss']:>7.4f} | {v['mean_iou']:>7.4f} | {v['mean_dice']:>7.4f} | {lr_now:>9.2e}")

    if v["mean_iou"] > best_iou:
        best_iou   = v["mean_iou"]
        no_improve = 0
        torch.save(model.state_dict(), WEIGHTS_PATH)
        print(f"         ✅ Saved best model (val IoU={best_iou:.4f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs).")
            break

print(f"\nBest Val IoU: {best_iou:.4f}")


## 📈 Training Curves

In [ ]:
epochs_ran = len(history["train"])
ep = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, key, title in zip(axes,
        ["loss", "mean_iou", "mean_dice"],
        ["Combined Loss", "Mean IoU", "Mean Dice"]):
    t_vals = [h[key] for h in history["train"]]
    v_vals = [h[key] for h in history["val"]]
    ax.plot(ep, t_vals, label="Train", linewidth=2)
    ax.plot(ep, v_vals, label="Val",   linewidth=2, linestyle="--")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Training History", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


## 🧪 Module 4: Final Test Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(dim=1)
        all_preds.append(preds.cpu())
        all_targets.append(masks)

all_preds   = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

metrics = compute_metrics(all_preds, all_targets)

print("\n" + "="*55)
print("       FINAL TEST SET RESULTS")
print("="*55)
print(f"  Pixel Accuracy : {metrics['pixel_acc']*100:.2f}%")
print(f"  Mean IoU       : {metrics['mean_iou']*100:.2f}%")
print(f"  Mean Dice      : {metrics['mean_dice']*100:.2f}%")
print("-"*55)
print(f"  {'Class':<10} {'IoU':>8} {'Dice':>8} {'Precision':>10} {'Recall':>8}")
print("-"*55)
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<10} {metrics['iou'][i]*100:>7.2f}% {metrics['dice'][i]*100:>7.2f}% "
          f"{metrics['precision'][i]*100:>9.2f}% {metrics['recall'][i]*100:>7.2f}%")
print("="*55)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(all_targets.flatten().numpy(),
                      all_preds.flatten().numpy(),
                      labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in zip(axes, [cm, cm_norm], ["Raw Counts", "Normalized"]):
    disp = ConfusionMatrixDisplay(confusion_matrix=data, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap="Blues",
              values_format=".0f" if title == "Raw Counts" else ".2f")
    ax.set_title(f"Confusion Matrix — {title}", fontsize=12, fontweight="bold")

plt.tight_layout(); plt.show()


## 🖼️ Module 5: Qualitative Visualization

In [ ]:
mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def tensor_to_rgb(t):
    return ((t * std_t + mean_t).permute(1,2,0).clamp(0,1).numpy())

def mask_to_color(mask_np):
    color = np.zeros((*mask_np.shape, 3))
    for c, col in enumerate(CLASS_COLORS):
        color[mask_np == c] = np.array(col)/255
    return color

# Pick samples with Oil present
model.eval()
n_show = 6
fig, axes = plt.subplots(4, n_show, figsize=(20, 14))

shown = 0
with torch.no_grad():
    for imgs, masks in test_loader:
        logits = model(imgs.to(DEVICE))
        preds  = logits.argmax(dim=1).cpu()
        for i in range(len(imgs)):
            if shown >= n_show: break
            if (masks[i] == 0).any():   # only show oil-present samples
                img_np   = tensor_to_rgb(imgs[i])
                gt_np    = mask_to_color(masks[i].numpy())
                pred_np  = mask_to_color(preds[i].numpy())
                overlay  = img_np.copy()
                oil_mask = (preds[i].numpy() == 0)
                overlay[oil_mask] = overlay[oil_mask] * 0.4 + np.array([1,0,0]) * 0.6

                axes[0, shown].imshow(img_np);   axes[0, shown].set_title(f"Input {shown+1}"); axes[0, shown].axis("off")
                axes[1, shown].imshow(gt_np);    axes[1, shown].set_title("Ground Truth");      axes[1, shown].axis("off")
                axes[2, shown].imshow(pred_np);  axes[2, shown].set_title("Prediction");        axes[2, shown].axis("off")
                axes[3, shown].imshow(overlay);  axes[3, shown].set_title("Oil Overlay");       axes[3, shown].axis("off")
                shown += 1
        if shown >= n_show: break

patches = [mpatches.Patch(color=np.array(c)/255, label=n)
           for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=12)
plt.suptitle("Qualitative Results — Test Set (Oil-present samples)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
x = np.arange(NUM_CLASSES)
w = 0.2
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(x - 1.5*w, [v*100 for v in metrics["iou"]],       w, label="IoU",       color="#2196F3")
ax.bar(x - 0.5*w, [v*100 for v in metrics["dice"]],      w, label="Dice",      color="#4CAF50")
ax.bar(x + 0.5*w, [v*100 for v in metrics["precision"]], w, label="Precision", color="#FF9800")
ax.bar(x + 1.5*w, [v*100 for v in metrics["recall"]],    w, label="Recall",    color="#9C27B0")

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_ylabel("Score (%)"); ax.set_ylim(0, 105)
ax.set_title("Per-Class Metrics — Test Set", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3)

for bars in ax.containers:
    ax.bar_label(bars, fmt="%.1f", fontsize=8, padding=2)

plt.tight_layout(); plt.show()


## 💾 Module 6: Export for Deployment

In [ ]:
# Save full model config for Streamlit app
import json

config = {
    "encoder":      "efficientnet-b3",
    "num_classes":  NUM_CLASSES,
    "class_names":  CLASS_NAMES,
    "class_colors": CLASS_COLORS,
    "img_size":     IMG_SIZE,
    "mean":         [0.485, 0.456, 0.406],
    "std":          [0.229, 0.224, 0.225],
    "metrics": {
        "pixel_accuracy": round(metrics["pixel_acc"]*100, 2),
        "mean_iou":       round(metrics["mean_iou"]*100, 2),
        "mean_dice":      round(metrics["mean_dice"]*100, 2),
        "per_class_iou":  [round(v*100, 2) for v in metrics["iou"]],
        "per_class_dice": [round(v*100, 2) for v in metrics["dice"]],
    }
}

config_path = "/content/drive/MyDrive/oilspill_model_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Model weights saved to:", WEIGHTS_PATH)
print("Config saved to:", config_path)
print("\nFinal Metrics Summary:")
print(f"  Pixel Accuracy : {config['metrics']['pixel_accuracy']}%")
print(f"  Mean IoU       : {config['metrics']['mean_iou']}%")
print(f"  Mean Dice      : {config['metrics']['mean_dice']}%")


## 🚀 Module 6: Streamlit Deployment App

In [ ]:
streamlit_code = '''
import streamlit as st
import torch, cv2, json, numpy as np
from PIL import Image
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

st.set_page_config(page_title="Oil Spill Detector", page_icon="🛢️", layout="wide")
st.title("🛢️ Oil Spill Detection — RGB Drone Imagery")
st.markdown("Upload a drone image to detect oil spills using U-Net + EfficientNet-B3")

@st.cache_resource
def load_model():
    cfg = json.load(open("oilspill_model_config.json"))
    model = smp.Unet(encoder_name="efficientnet-b3", encoder_weights=None,
                     in_channels=3, classes=3, activation=None)
    model.load_state_dict(torch.load("best_oilspill_unet.pth", map_location="cpu"))
    model.eval()
    return model, cfg

model, cfg = load_model()

CLASS_COLORS = [(255,0,0),(0,0,255),(128,128,128)]
transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=cfg["mean"], std=cfg["std"]),
    ToTensorV2()
])

uploaded = st.file_uploader("Upload drone image (.png / .jpg)", type=["png","jpg","jpeg"])

if uploaded:
    img = np.array(Image.open(uploaded).convert("RGB"))
    aug = transform(image=img)
    inp = aug["image"].unsqueeze(0)
    with torch.no_grad():
        pred = model(inp).argmax(dim=1).squeeze().numpy()

    color_mask = np.zeros((*pred.shape, 3), dtype=np.uint8)
    for c, col in enumerate(CLASS_COLORS):
        color_mask[pred == c] = col

    overlay = cv2.resize(img, (256,256)).astype(float)
    oil_px  = (pred == 0)
    overlay[oil_px] = overlay[oil_px] * 0.4 + np.array([255,0,0]) * 0.6
    overlay = overlay.clip(0,255).astype(np.uint8)

    col1, col2, col3 = st.columns(3)
    col1.image(cv2.resize(img,(256,256)), caption="Input Image",    use_column_width=True)
    col2.image(color_mask,               caption="Segmentation Mask", use_column_width=True)
    col3.image(overlay,                  caption="Oil Overlay",      use_column_width=True)

    oil_pct = (pred==0).mean()*100
    st.metric("🔴 Oil Coverage", f"{oil_pct:.1f}%")
    if oil_pct > 5:
        st.error(f"⚠️ ALERT: Significant oil spill detected — {oil_pct:.1f}% of frame")
    elif oil_pct > 0:
        st.warning(f"Oil detected in {oil_pct:.1f}% of frame")
    else:
        st.success("✅ No oil detected")

    st.markdown("### Model Performance")
    m = cfg["metrics"]
    c1,c2,c3 = st.columns(3)
    c1.metric("Pixel Accuracy", f"{m[\'pixel_accuracy\']}%")
    c2.metric("Mean IoU",       f"{m[\'mean_iou\']}%")
    c3.metric("Mean Dice",      f"{m[\'mean_dice\']}%")
'''

with open("/content/drive/MyDrive/app.py", "w") as f:
    f.write(streamlit_code)

print("Streamlit app saved to Drive!")
print("\nTo run locally:")
print("  streamlit run app.py")
print("\nTo run in Colab:")
print("  !pip install streamlit pyngrok -q")
print("  !streamlit run app.py &")
